In [1]:
"""
Who Wins Where?
=========================
Goal: We trained a model on every PGA Tour and DP World Tour
      event from 2019-2025 to predict who wins where.
"""
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add scripts directory to path so we can import caddie_style
from pathlib import Path
ROOT = Path.cwd().parent.parent  # caddie-desk root
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import caddie_style
caddie_style.apply()

print("Setup complete.")
print(f"Working from: {ROOT}")

Setup complete.
Working from: /Users/rosie.kipling/Desktop/caddiedesk


<h2> Who wins <em>where?</em> </h2>

**Goal:** We trained a model on every PGA Tour and DP World Tour event from 2019-2025 to predict who wins where. Here's what it says about Birkdale.

<h3>Statistical signature (the gold) </h3>

1. **Stat-finish correlations** — for each of SG:OTT, SG:APP, SG:ARG, SG:PUT, plus driving distance and accuracy, how strongly does that stat correlate with winning at this course? Example: At a long bomber's course, SG:OTT correlation with finish position is high (better off-tee → better finish). At a short technical course, SG:APP dominates.
2. **Variance in stat importance** — how much do the top finishers differ from average? At Augusta, the gap between winner and field on SG:APP is huge. At a TPC course, less so.
3. **Stat consistency at the course** — does the same type of player win year after year? Some courses produce wildly different winners (Open Championship), others are consistent (Augusta).
4. **Scoring distribution** — how much spread between leader and cut line? Bunched leaderboards = course doesn't differentiate players; spread leaderboards = it does.
5. **Cut line vs par** — birdie-fest course or grinder course?

<h3> Physical/architectural (the colour) </h3>

Useful for narrative, harder to scrape, but worth attempting:

1. Course type: Links / parkland / desert / coastal / mountain
1. Surface type: Bentgrass, Bermuda, Poa annua, fescue (matters for putting style)
1. Wind exposure: Coastal vs sheltered
1. Length: Total yardage
1. Par: 70/71/72 (no.par 3's, no par 5s)
1. Greens architecture: Bentgrass speed, undulation, size
1. Rough penalty: Brutal rough vs forgiving rough
1. Fairway width: Tight vs wide
1. Tree-lined vs open: Affects strategic options
1. Elevation: Sea level vs mountain (ball flies further at altitude)
1. Climate during tournament week: Average temp, humidity, wind


Additional:
1. no. Doglegs left/right/straight
2. no. bunkers
3. country (indication of weather)

<h3> Building a model to predict which golfers do well on a specific course.</h3>

**A more rigorous reframe**

What you actually want is:

`(course features, player stats) → (player's finishing position at that course)`

Train this on ALL players in ALL events (not just winners). Once trained, you can do something clever at inference time:

For Birkdale specifically, you generate a population of "synthetic players" by varying their stat profiles across the full range. Run each synthetic player through the model with Birkdale's course features. See which synthetic profile gets the best predicted finishing position. That is the "ideal player profile" for Birkdale.

This is sometimes called "inverse modelling" or "profile optimisation."

<h4> Build the Prediction Model - XGBoost</h4>

Predict strokes vs par - Lots of signal, no class imbalance, doesn't lose info by binning.





In [2]:
# Inputs (per row = one player at one event):
features = [
    # Course features
    "course_yardage",
    "course_par",
    "fairway_width",
    "rough_penalty",
    "green_speed",
    "wind_avg",
    "scoring_avg",        # statistical signature from earlier clustering
    
    # Player features (their season-long stats heading into this event)
    "player_sg_ott",
    "player_sg_app",
    "player_sg_arg",
    "player_sg_putt",
    "player_driving_dist",
    "player_driving_acc",
    "player_world_rank",  # general skill control
]

# Target:
target = "finish_position"  # or strokes vs par, or made_cut, etc.

<h4>Find the optimal player for Any Course</h4>

In [ ]:
def find_optimal_profile(course_features, model, player_stat_ranges):
    """
    Given a course, find the player stat profile that minimises predicted finish.
    """
    # Option A: Grid search over plausible player stat ranges
    best_profile = None
    best_predicted_finish = float('inf')
    
    for sg_ott in np.linspace(-1, 2.5, 20):
        for sg_app in np.linspace(-0.5, 2.5, 20):
            for sg_arg in np.linspace(-0.5, 1.5, 15):
                for sg_putt in np.linspace(-0.5, 1.5, 15):
                    profile = {
                        "sg_ott": sg_ott,
                        "sg_app": sg_app,
                        "sg_arg": sg_arg,
                        "sg_putt": sg_putt,
                    }
                    features = {**course_features, **profile}
                    pred = model.predict(features)
                    if pred < best_predicted_finish:
                        best_predicted_finish = pred
                        best_profile = profile
    
    return best_profile

<h4>Find real players closest to that profile</h4>


In [ ]:
def closest_players(ideal_profile, current_players, top_n=10):
    distances = []
    for player in current_players:
        profile_vec = np.array([player['sg_ott'], player['sg_app'], 
                                player['sg_arg'], player['sg_putt']])
        ideal_vec = np.array([ideal_profile['sg_ott'], ideal_profile['sg_app'],
                              ideal_profile['sg_arg'], ideal_profile['sg_putt']])
        dist = np.linalg.norm(profile_vec - ideal_vec)
        distances.append((player['name'], dist))
    
    return sorted(distances, key=lambda x: x[1])[:top_n]